# Preprocessing Pipeline

Turns the findings from `01_eda.ipynb` into a model-ready dataset and a fitted, reusable preprocessing pipeline.

In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
DATA_DIR = "../data/raw"

df = pd.read_csv(f"{DATA_DIR}/application_train.csv")
df.shape

(307511, 122)

## 1. Cleaning

Fixed rules only: known bad values removed, a documented placeholder recoded, columns dropped on criteria the EDA already settled.

### 1a. Dropping columns

Three rules:
- **Above 50% missing** — too little left to impute from. One exception: `EXT_SOURCE_1`.
- **Redundant copies** — each building measurement appears three times as `_AVG`/`_MEDI`/`_MODE`; only `_AVG` is kept. `TOTALAREA_MODE` and `EMERGENCYSTATE_MODE` are spared — neither has any duplicates.
- **`SK_ID_CURR`** — an application reference number, with no predictive content.

In [2]:
MISSING_THRESHOLD = 50

features = df.drop(columns="TARGET")
missing_pct = features.isna().mean() * 100
too_empty = set(missing_pct[missing_pct > MISSING_THRESHOLD].index) - {"EXT_SOURCE_1"}

# a _MEDI or _MODE column is redundant only when the same measurement also exists as _AVG
avg_measurements = {c[:-4] for c in features.columns if c.endswith("_AVG")}
redundant = {
    c for c in features.columns
    if c.endswith(("_MEDI", "_MODE")) and c[:-5] in avg_measurements
}

drop_cols = sorted(too_empty | redundant | {"SK_ID_CURR"})
df = df.drop(columns=drop_cols)

print(f"more than {MISSING_THRESHOLD}% empty : {len(too_empty):>3}")
print(f"redundant copies      : {len(redundant):>3}")
print(f"identifier            : {1:>3}")
print(f"dropped in total      : {len(drop_cols):>3}   (the first two rules overlap)")
print(f"columns remaining     : {df.shape[1]:>3}   (including TARGET)")

kept_empty = [c for c in df.columns if df[c].isna().mean() > 0.5]
print(f"\nstill above {MISSING_THRESHOLD}% missing, by choice: {kept_empty}")

more than 50% empty :  40
redundant copies      :  28
identifier            :   1
dropped in total      :  45   (the first two rules overlap)
columns remaining     :  77   (including TARGET)

still above 50% missing, by choice: ['EXT_SOURCE_1']


### 1b. Dropping rows

`CODE_GENDER = 'XNA'` (4 rows) and `NAME_FAMILY_STATUS = 'Unknown'` (2 rows) are placeholder categories; `AMT_INCOME_TOTAL = 117,000,000` (1 row) is implausible against a median of 147,150.

In [3]:
n_before = len(df)

df = df[
    (df["CODE_GENDER"] != "XNA")
    & (df["NAME_FAMILY_STATUS"] != "Unknown")
    & (df["AMT_INCOME_TOTAL"] != 117_000_000)
].reset_index(drop=True)

print(f"dropped {n_before - len(df)} rows ({n_before:,} -> {len(df):,})")

dropped 7 rows (307,511 -> 307,504)


### 1c. The `DAYS_EMPLOYED` placeholder

The column is documented as *days before the application that the client started current employment*, so every genuine value is negative. About 18% of rows hold `365243` instead — positive, roughly a thousand years — and those rows are pensioners plus every unemployed client in the data.

The definition presupposes current employment, so replace it with `0` would claim the client started work on the application date. By contrast, `NaN` is the only value meaning "not applicable".

This value must be replaced before performing imputation. Otherwise, the large number of occurrences of this code will distort the median.

In [4]:
placeholder = df["DAYS_EMPLOYED"] == 365243

print(f"placeholder rows: {placeholder.sum():,} ({placeholder.mean():.1%})")
print(df.loc[placeholder, "NAME_INCOME_TYPE"].value_counts().to_string())

df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

print(f"\nDAYS_EMPLOYED is now {df['DAYS_EMPLOYED'].isna().mean():.1%} missing, to be filled at imputation")

placeholder rows: 55,374 (18.0%)
NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22

DAYS_EMPLOYED is now 18.0% missing, to be filled at imputation


## 2. Feature engineering

Both features below are arithmetic within a single row, involving no statistic pooled across clients, so they still precede the split.

### 2a. Converting the `DAYS_` columns to years

In [5]:
DAYS_TO_YEARS = {
    "DAYS_BIRTH": "AGE_YEARS",
    "DAYS_EMPLOYED": "EMPLOYMENT_YEARS",
    "DAYS_REGISTRATION": "YEARS_SINCE_REGISTRATION",
    "DAYS_ID_PUBLISH": "YEARS_SINCE_ID_PUBLISH",
    "DAYS_LAST_PHONE_CHANGE": "YEARS_SINCE_PHONE_CHANGE",
}

# every value must already be a count backwards, or negating would silently flip it
assert (df[list(DAYS_TO_YEARS)].max() <= 0).all(), "a DAYS_ column holds a positive value"

years = pd.DataFrame({new: df[old].abs() / 365.25 for old, new in DAYS_TO_YEARS.items()})
df = pd.concat([df.drop(columns=list(DAYS_TO_YEARS)), years], axis=1)

print(df[list(DAYS_TO_YEARS.values())].describe().round(2).to_string())

       AGE_YEARS  EMPLOYMENT_YEARS  YEARS_SINCE_REGISTRATION  YEARS_SINCE_ID_PUBLISH  YEARS_SINCE_PHONE_CHANGE
count  307504.00         252130.00                 307504.00               307504.00                 307503.00
mean       43.91              6.53                     13.65                    8.20                      2.64
std        11.95              6.40                      9.65                    4.13                      2.26
min        20.50              0.00                      0.00                    0.00                      0.00
25%        33.98              2.10                      5.50                    4.71                      0.75
50%        43.12              4.51                     12.33                    8.91                      2.07
75%        53.89              8.69                     20.48                   11.77                      4.30
max        69.07             49.04                     67.55                   19.70                     11.75


### 2b. Ratio features

Lending is underwritten on capacity to repay: the same 500,000 loan is routine for one
applicant and unpayable for another. `AMT_CREDIT` alone cannot tell them apart. Two standard
affordability measures can — debt-to-income and payment-to-income:

- `CREDIT_INCOME_RATIO` — the loan in years of gross income.
- `ANNUITY_INCOME_RATIO` — the share of annual income each instalment consumes.

The source columns stay: a ratio discards the levels, so 100k on 50k of income and 400k on
200k both score 2.0.

In [6]:
ratios = pd.DataFrame({
    "CREDIT_INCOME_RATIO": df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"],
    "ANNUITY_INCOME_RATIO": df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"],
})
df = pd.concat([df, ratios], axis=1)

print(df[["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO"]].describe().round(2).to_string())
print(f"\nfinal shape: {df.shape}")

       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO
count            307504.00             307492.00
mean                  3.96                  0.18
std                   2.69                  0.09
min                   0.04                  0.00
25%                   2.02                  0.11
50%                   3.27                  0.16
75%                   5.16                  0.23
max                  84.74                  1.88

final shape: (307504, 79)


The correlation structure among the money columns, with the ratios now included.

In [7]:
money_cols = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO",
]
df[money_cols].corr().round(2)

,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO
AMT_INCOME_TOTAL,1.00,0.34,0.42,0.35,-0.23,-0.33
AMT_CREDIT,0.34,1.00,0.77,0.99,0.65,0.37
AMT_ANNUITY,0.42,0.77,1.00,0.78,0.39,0.48
AMT_GOODS_PRICE,0.35,0.99,0.78,1.00,0.63,0.37
CREDIT_INCOME_RATIO,-0.23,0.65,0.39,0.63,1.00,0.79
ANNUITY_INCOME_RATIO,-0.33,0.37,0.48,0.37,0.79,1.00


`AMT_CREDIT` and `AMT_GOODS_PRICE` correlate at 0.99 — the loan is almost always the price of the financed goods — so `AMT_GOODS_PRICE` is dropped.

In [8]:
df = df.drop(columns="AMT_GOODS_PRICE")

remaining = [c for c in money_cols if c in df.columns]
print(f"dropped AMT_GOODS_PRICE -> {df.shape[1]} columns")

dropped AMT_GOODS_PRICE -> 78 columns


### 2c. Missingness indicator

Columns with more than 20% missing get an additional `IS_MISSING_COLUMN` flag before imputation. This flag preserves the "data unavailable" signal that may be informative for the model. After the flag is created, missing values for numeric columns are imputed with median, while for categorical columns a "Missing" category is added.

`TOTALAREA_MODE`, `FLOORSMAX_AVG` and `YEARS_BEGINEXPLUATATION_AVG` are mostly missing together, so only one indicator is created for all of them.

The categorical columns need nothing. `SimpleImputer(strategy="constant", fill_value="Missing")` gives them an explicit `Missing` level, and once one-hot encoded that level *is* a missingness indicator.

In [9]:
cols = ['TOTALAREA_MODE', 'FLOORSMAX_AVG', 'YEARS_BEGINEXPLUATATION_AVG']
check_miss = df[cols].isna().all(axis=1).sum() / df[cols].isna().any(axis=1).sum() * 100
print(f"{check_miss:.1f}% of the rows that have missing data in at least one of these three columns have missing data in all three columns.")

96.2% of the rows that have missing data in at least one of these three columns have missing data in all three columns.


In [10]:
INDICATE_MISSING = ["EXT_SOURCE_1", "TOTALAREA_MODE"]

indicators = pd.DataFrame({
    f"IS MISSING_{col}": df[col].isna().astype(int) for col in INDICATE_MISSING
})
df = pd.concat([df, indicators], axis=1)

print(f"\nshape: {df.shape}")


shape: (307504, 80)


## 3. Train / validation / test split

The data is cut 70/15/15 (train/val/test), stratified on the target so all three sets carry the same 8.07%

In [11]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.15
VAL_SIZE = 0.15

X = df.drop(columns="TARGET")
y = df["TARGET"]

# Step 1: train-test split (85%-15%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# Step 2: from the train set (85%), separate out val (~17.65% of 85% ≈ 15% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    stratify=y_trainval,
    random_state=RANDOM_STATE,
)

for name, y_part in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:<6}{len(y_part):>8,} rows   {len(y_part)/len(y):>5.1%} of total   "
          f"default rate {y_part.mean():.4f}")

train  215,252 rows   70.0% of total   default rate 0.0807
val     46,126 rows   15.0% of total   default rate 0.0807
test    46,126 rows   15.0% of total   default rate 0.0807


## 4. Building the preprocessor

Four column groups, each with its own treatment, assembled into a single `ColumnTransformer`: 79 columns in, 146 features out.

### 4a. Column groups

`FLAG_OWN_CAR` and `FLAG_OWN_REALTY` are named like the 26 binary `FLAG_DOCUMENT_*` columns, but hold `Y`/`N` text; selecting on the prefix would send text into a numeric branch. Grouping is therefore by dtype, not by name. Nothing is lost: one-hot encoding a two-valued column with `drop="first"` returns the same 1/0.

**`NAME_EDUCATION_TYPE` is separated out** because its categories have a real order that one-hot encoding would discard. The ordering is verified against default rate below rather than assumed.

In [12]:
ORDINAL_COLS = ["NAME_EDUCATION_TYPE"]

text_cols = X_train.select_dtypes(exclude="number").columns.tolist()

categorical_cols = [c for c in text_cols if c not in ORDINAL_COLS]
flag_cols = [c for c in X_train.columns if c.startswith("FLAG_") and c not in text_cols]
numeric_cols = [c for c in X_train.columns if c not in text_cols + flag_cols]

print(f"{'ordinal':<14}{len(ORDINAL_COLS):>4}")
print(f"{'categorical':<14}{len(categorical_cols):>4}")
print(f"{'binary flags':<14}{len(flag_cols):>4}")
print(f"{'numeric':<14}{len(numeric_cols):>4}")

ordinal          1
categorical     12
binary flags    26
numeric         40


In [13]:
EDUCATION_ORDER = [
    "Lower secondary",
    "Secondary / secondary special",
    "Incomplete higher",
    "Higher education",
    "Academic degree",
]

by_education = y_train.groupby(X_train["NAME_EDUCATION_TYPE"]).agg(["mean", "size"])
by_education = by_education.reindex(EDUCATION_ORDER)
by_education["mean"] = (by_education["mean"] * 100).round(2)

print(by_education.rename(columns={"mean": "default_%", "size": "n"}).to_string())

                               default_%       n
NAME_EDUCATION_TYPE                             
Lower secondary                    11.03    2683
Secondary / secondary special       8.94  152903
Incomplete higher                   8.53    7224
Higher education                    5.34   52327
Academic degree                     2.61     115


### 4b. The numeric branch

Two learned operations: a median to fill gaps, a mean and standard deviation to scale.

**Scaling** matters for the linear models: a regularisation penalty treats all coefficients alike, so an unscaled feature measured in hundreds of thousands would be penalised out for reasons unrelated to its value. The tree models are unaffected either way.

**Imputation is the median, uniformly.** `EMPLOYMENT_YEARS` is 18% missing and those gaps are pensioners, who therefore receive about four and a half years of employment. The fact survives in `NAME_INCOME_TYPE`, so it is lost from the column rather than from the dataset.

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_branch = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

### 4c. The text branches

**Ordinal — `NAME_EDUCATION_TYPE`** → 0–4 in the verified order, from an explicit list rather than alphabetically. No missing values, so no imputer. Scaled with everything else, or it would be the one feature on a different scale under a penalty.

**Nominal — the other 12** → one 0/1 column per category, with four settings that matter:
- `SimpleImputer(constant, "Missing")` — absence as a category. `OCCUPATION_TYPE` is 31% blank, largely pensioners; a mode would invent a job.
- `min_frequency=800` — merges categories with less than 800 clients. `ORGANIZATION_TYPE` has 26 such, covering 3.5% of clients.
- `handle_unknown="infrequent_if_exist"` — an unseen category at prediction time joins that bucket instead of raising.
- `drop="first"` — drops one level per column as the reference, so the dummies are not collinear with the intercept.

In [15]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

ordinal_branch = Pipeline([
    ("encode", OrdinalEncoder(
        categories=[EDUCATION_ORDER],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )),
    ("scale", StandardScaler()),
])

categorical_branch = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encode", OneHotEncoder(
        min_frequency=800,
        handle_unknown="infrequent_if_exist",
        drop="first",
        sparse_output=False,
    )),
])

# where min_frequency actually merged categories
categorical_branch.fit(X_train[categorical_cols])
encoder = categorical_branch.named_steps["encode"]

merged = pd.DataFrame(
    [
        {"categories": len(cats), "merged": len(infrequent), "columns out": len(cats) - len(infrequent)}
        for cats, infrequent in zip(encoder.categories_, encoder.infrequent_categories_)
        if infrequent is not None
    ],
    index=[c for c, inf in zip(categorical_cols, encoder.infrequent_categories_) if inf is not None],
).sort_values("merged", ascending=False)

print(f"min_frequency=800 merged rare categories in {len(merged)} of {len(categorical_cols)} columns:")
print(merged.to_string())

min_frequency=800 merged rare categories in 5 of 12 columns:
                   categories  merged  columns out
ORGANIZATION_TYPE          58      26           32
NAME_INCOME_TYPE            8       4            4
OCCUPATION_TYPE            19       3           16
NAME_TYPE_SUITE             8       2            6
NAME_HOUSING_TYPE           6       1            5


### 4d. Assembly

`ColumnTransformer` routes each group to its branch and concatenates the results; the flags pass through unchanged. Unlisted columns are dropped silently, so an assertion checks the four groups cover every column.

`set_output(transform="pandas")` keeps real feature names — `numeric__AMT_CREDIT` rather than a position. 

In [16]:
from sklearn.compose import ColumnTransformer

# nothing may fall through: the four groups must account for every column
routed = numeric_cols + ORDINAL_COLS + categorical_cols + flag_cols
assert set(routed) == set(X_train.columns), (
    f"unrouted columns would be dropped: {set(X_train.columns) - set(routed)}"
)

preprocessor = ColumnTransformer([
    ("numeric", numeric_branch, numeric_cols),
    ("ordinal", ordinal_branch, ORDINAL_COLS),
    ("categorical", categorical_branch, categorical_cols),
    ("flags", "passthrough", flag_cols),
]).set_output(transform="pandas")

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``f

## 5. Fitting and saving

### 5a. Fit and transform

Fitted on the training fold only; validation and test are transformed with the values it learned there.

In [17]:
X_train_prep = preprocessor.fit_transform(X_train)
X_val_prep = preprocessor.transform(X_val)
X_test_prep = preprocessor.transform(X_test)

for name, before, after in [
    ("train", X_train, X_train_prep),
    ("val", X_val, X_val_prep),
    ("test", X_test, X_test_prep),
]:
    print(f"{name:<6}{before.shape[1]:>4} -> {after.shape[1]:>4} features   "
          f"{int(before.isna().sum().sum()):>8,} gaps -> {int(after.isna().sum().sum())}")

train   79 ->  146 features    866,778 gaps -> 0
val     79 ->  146 features    183,997 gaps -> 0
test    79 ->  146 features    187,699 gaps -> 0


### 5b. Saving the artifacts

**The cleaned split**, so fold membership is fixed permanently — editing a cleaning rule later would otherwise reshuffle the folds and invalidate any result already reported.

**The fitted preprocessor**, holding the medians, category lists and scaling parameters. It is what lets a model be fed exactly what the pipeline was fitted against, and what lets a single new applicant be scored.

In [18]:
import os
import joblib

PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

for name, X_part, y_part in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test),
]:
    joblib.dump(X_part.assign(TARGET=y_part.values), f"{PROCESSED_DIR}/{name}.joblib")

joblib.dump(preprocessor, f"{MODELS_DIR}/preprocessor.joblib")

for folder in (PROCESSED_DIR, MODELS_DIR):
    for f in sorted(os.listdir(folder)):
        size_kb = os.path.getsize(f"{folder}/{f}") / 1e3
        size, unit = (size_kb / 1e3, "MB") if size_kb >= 1e3 else (size_kb, "KB")
        print(f"{folder + '/' + f:<34}{size:>7.1f} {unit}")

../data/processed/test.joblib        26.7 MB
../data/processed/train.joblib      124.5 MB
../data/processed/val.joblib         26.7 MB
../models/preprocessor.joblib        30.8 KB


In [19]:
reloaded = joblib.load(f"{MODELS_DIR}/preprocessor.joblib")
reloaded_output = reloaded.transform(X_test)

print("reloads and reproduces its output exactly:",
      list(reloaded_output.columns) == list(X_test_prep.columns)
      and (reloaded_output.to_numpy() == X_test_prep.to_numpy()).all())

reloads and reproduces its output exactly: True
